<a href="https://colab.research.google.com/github/mitalidaduria/enterprise-data-platform/blob/main/Execution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!rm -rf enterprise-data-platform
!git clone https://github.com/mitalidaduria/enterprise-data-platform.git
import os
os.chdir("enterprise-data-platform")
print("📂 Current Directory:", os.getcwd())

Cloning into 'enterprise-data-platform'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 62 (delta 16), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 30.72 KiB | 3.84 MiB/s, done.
Resolving deltas: 100% (16/16), done.
📂 Current Directory: /content/enterprise-data-platform


In [2]:
!pip install pyspark -q

print("1️⃣ Generating raw synthetic data...")
!python pipelines/generate_data.py

print("2️⃣ Running PySpark ingestion & PII hashing...")
!python pipelines/ingest_pyspark.py

print("3️⃣ Executing Data Quality Gates & Quarantine...")
!python pipelines/data_quality.py

print("4️⃣ Running MDM Entity Resolution & Golden Record generation...")
!python pipelines/entity_resolution.py

print("\n🎉 All pipeline stages completed successfully!")

1️⃣ Generating raw synthetic data...
  File "/content/enterprise-data-platform/pipelines/generate_data.py", line 35
    raw_email = f"{first_name.lower()}.{last_name.lower()}{i}@"{domain}
                                                               ^
SyntaxError: invalid syntax
2️⃣ Running PySpark ingestion & PII hashing...
[0.034s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.034s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Using Spark's 

In [4]:
# 1. Overwrite pipelines/generate_data.py with clean, bug-free code
clean_generator_code = '''"""
Enterprise Data Platform: Raw Synthetic Data Generator
Generates mock multi-source operational datasets (Billing & Shipping)
with realistic PII, typos, and overlapping entity attributes for MDM testing.
"""

import csv
import hashlib
import os
import random
import uuid

DATA_DIR = os.path.join(os.path.dirname(__file__), "raw_data")
os.makedirs(DATA_DIR, exist_ok=True)

NUM_RECORD_PAIRS = 1000

FIRST_NAMES = ["Robert", "Bob", "Rob", "William", "Bill", "Elizabeth", "Liz", "Michael", "Mike", "Sarah"]
LAST_NAMES = ["Jones", "Smith", "Taylor", "Brown", "Wilson", "Davies", "Evans", "Thomas", "Johnson"]
STREETS = ["123 Main St", "456 Oak Ave", "789 Pine Rd", "101 Maple Dr", "202 Birch Ln"]
DOMAINS = ["gmail.com", "yahoo.com", "hotmail.com", "enterprise.org"]

def generate_datasets():
    billing_rows = []
    shipping_rows = []

    for i in range(NUM_RECORD_PAIRS):
        first_name = random.choice(FIRST_NAMES)
        last_name = random.choice(LAST_NAMES)
        full_name = f"{first_name} {last_name}"
        domain = random.choice(DOMAINS)

        raw_email = f"{first_name.lower()}.{last_name.lower()}{i}@{domain}"

        billing_id = f"BIL-{uuid.uuid4().hex[:8].upper()}"
        credit_card_hash = hashlib.sha256(f"CARD-{i}".encode()).hexdigest()
        billing_address = random.choice(STREETS)

        billing_rows.append([
            billing_id,
            full_name,
            raw_email,
            credit_card_hash,
            billing_address
        ])

        shipping_id = f"SHP-{uuid.uuid4().hex[:8].upper()}"
        recipient_name = f"{first_name[0]}. {last_name}"
        phone_number = f"555-{random.randint(100, 999)}-{random.randint(1000, 9999)}"
        street_address = billing_address

        shipping_rows.append([
            shipping_id,
            recipient_name,
            raw_email,
            street_address,
            phone_number
        ])

    billing_path = os.path.join(DATA_DIR, "raw_billing.csv")
    with open(billing_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["billing_id", "full_name", "email", "credit_card_hash", "billing_address"])
        writer.writerows(billing_rows)

    shipping_path = os.path.join(DATA_DIR, "raw_shipping.csv")
    with open(shipping_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["shipping_id", "recipient_name", "email", "street_address", "phone_number"])
        writer.writerows(shipping_rows)

    print(f"✅ Generated {NUM_RECORD_PAIRS} synthetic raw records successfully.")

if __name__ == "__main__":
    generate_datasets()
'''

with open("pipelines/generate_data.py", "w") as f:
    f.write(clean_generator_code)

print("✅ Fixed generate_data.py! Now running full pipeline...")

# 2. Run the full pipeline sequentially
import subprocess
subprocess.run(["python", "pipelines/generate_data.py"], check=True)
subprocess.run(["python", "pipelines/ingest_pyspark.py"], check=True)
subprocess.run(["python", "pipelines/data_quality.py"], check=True)
subprocess.run(["python", "pipelines/entity_resolution.py"], check=True)
subprocess.run(["python", "infrastructure/load_db.py"], check=True)

print("\n✨ Pipeline execution complete! Testing API Client...")

# 3. Test API Gateway
from fastapi.testclient import TestClient
from gateway.app import app

client = TestClient(app)
res = client.get("/customers?limit=3")
print("\n🔥 STATUS CODE:", res.status_code)
print("✨ DATA:", res.json())

✅ Fixed generate_data.py! Now running full pipeline...

✨ Pipeline execution complete! Testing API Client...

🔥 STATUS CODE: 200
✨ DATA: [{'golden_customer_id': '35508333febf1ddc32a8941d9a694274ec99dee44e334236f49ac2a2c9cf5607', 'primary_name': 'Sarah Evans', 'primary_email_hash': '004fccf9d4bdac7ecc346ff855a720a8b6dd2a7b0be7ff2911e4cca369538d10', 'primary_phone': '555-380-9240', 'primary_address': '202 Birch Ln', 'billing_linkage_id': 'BIL-3C09B869', 'shipping_linkage_id': 'SHP-A949B6E3', 'total_source_linkages': 2, 'pipeline_version': 'v1.0.0'}, {'golden_customer_id': '23c90512dedeff5f41785f3e9cc0166a0c6eb30f7470a989d9fd47edfefa1c99', 'primary_name': 'Elizabeth Wilson', 'primary_email_hash': '0080ecc0cac5e3c6cb0ae5e53d5cc3e0f6308b10baa6647d0b6103a8d260e873', 'primary_phone': '555-587-8219', 'primary_address': '123 Main St', 'billing_linkage_id': 'BIL-F0798E8C', 'shipping_linkage_id': 'SHP-9D27688A', 'total_source_linkages': 2, 'pipeline_version': 'v1.0.0'}, {'golden_customer_id': '3f